# Feature Engineering - UCI Bank Marketing Dataset

**CAP182 SS2 Capstone – STADIO Equities First-Deposit Activation** 
**Stage:** Feature Engineering (Stage 2 of the data science workflow)
**Input:** `data/processed/bank_marketing_preprocessed.csv` (output of the preprocessing stage)

---

This notebook prepares a **modelling-ready predictor and target matrix**. It does not train,
tune, cross-validate or evaluate any model. Those steps belong to the modelling stage,
in which will use Logistic Regression and Gradient Boosting on the output of this
notebook.


## 1. Load the Preprocessed Dataset

The preprocessing stage prepared the raw UCI Bank Marketing dataset by investigating data quality issues and applying the agreed preprocessing decisions. The resulting dataset was saved as `bank_marketing_preprocessed.csv`.

This notebook uses that processed dataset as its starting point for feature engineering. At this stage, the purpose is to confirm that the correct dataset has been loaded before creating new modelling features.

The UCI Bank Marketing dataset is used as a structural proxy for the STADIOEquities activation problem. The target variable represents whether a client subscribed to a term deposit and is therefore used as a proxy for the project's financial conversion outcome. The dataset is not used to make claims about actual STADIOEquities customers.

In [1]:
import pandas as pd
import numpy as np

INPUT_PATH = "data/processed/bank_marketing_preprocessed.csv"

df = pd.read_csv(INPUT_PATH)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully.
Rows: 45211
Columns: 18


In [2]:
#Confirming the input variables
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Column names:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y', 'y_binary']

Data types:
age            int64
job           object
marital       object
education     object
default       object
balance        int64
housing       object
loan          object
contact       object
day            int64
month         object
duration       int64
campaign       int64
pdays        float64
previous       int64
poutcome      object
y             object
y_binary       int64
dtype: object


## 2. Verify the Target and Preprocessing Decisions

Before creating new features, the key preprocessing decisions are verified to ensure that the feature-engineering stage is working from the expected data.

The target variable `y_binary` should contain only two classes: 0 and 1. The `pdays` variable should no longer contain the original `-1` value. Instead, customers who were not previously contacted are represented by missing values, which will be transformed into an explicit indicator feature in the next step.

In [3]:
# Check the number of observations in each target class
print("Target distribution:")
print(df["y_binary"].value_counts())

# Check that the target contains only the expected binary values
print("\nUnique target values:")
print(df["y_binary"].unique())

# Confirm that the original -1 special value has been removed from pdays
print("\nNumber of -1 values in pdays:")
print((df["pdays"] == -1).sum())

# Count the missing pdays values created during preprocessing
print("\nNumber of missing pdays values:")
print(df["pdays"].isna().sum())

Target distribution:
y_binary
0    39922
1     5289
Name: count, dtype: int64

Unique target values:
[0 1]

Number of -1 values in pdays:
0

Number of missing pdays values:
36954


## 3. Create Engineered Features

Feature engineering transforms existing variables into additional features that may provide useful information for the modelling stage.

The engineered features are created from the preprocessing output and are based on the characteristics of the dataset and the project's objective of identifying patterns associated with financial conversion.

**3.1 Create the Previously Contacted Feature**

The `pdays` variable records the number of days since a customer was last contacted during a previous campaign. In the preprocessing stage, the value `-1` was converted to a missing value because it indicates that the customer had not previously been contacted rather than representing a meaningful number of days.

A binary feature called `previously_contacted` is therefore created. It records whether a previous contact existed:

- `1` = customer had been contacted previously
- `0` = customer had not been contacted previously

This converts the special-value information in `pdays` into an explicit feature that can be used during modelling.

In [6]:
# Create a binary indicator showing whether a previous contact exists
df["previously_contacted"] = df["pdays"].notna().astype(int)

# Check the number of customers in each category
print("Previously contacted distribution:")
print(df["previously_contacted"].value_counts())

# Confirm that the new feature contains only 0 and 1
print("\nUnique values:")
print(df["previously_contacted"].unique())

Previously contacted distribution:
previously_contacted
0    36954
1     8257
Name: count, dtype: int64

Unique values:
[0 1]


**3.2 Create Age Bands**

The original `age` variable is retained as a numerical variable. In addition, age bands are created to represent broader age groups.

This allows the modelling stage to investigate whether conversion patterns differ across age ranges without assuming that the relationship between age and the target is strictly linear.

The age bands are created as:

- 18–29
- 30–39
- 40–49
- 50–59
- 60+

In [7]:
# Define age ranges and assign each customer to an age band
df["age_band"] = pd.cut(
    df["age"],
    bins=[17, 29, 39, 49, 59, np.inf],
    labels=["18-29", "30-39", "40-49", "50-59", "60+"]
)

# Check how many customers fall into each age band
print("Age band distribution:")
print(df["age_band"].value_counts().sort_index())

Age band distribution:
age_band
18-29     5273
30-39    18089
40-49    11655
50-59     8410
60+       1784
Name: count, dtype: int64


**3.3 Create Balance Bands**

The `balance` variable represents the customer's account balance. The original numerical variable is retained, while balance bands are created to represent broader balance ranges.

This provides an additional way of representing the customer's financial position and allows the modelling stage to investigate whether conversion patterns differ across balance ranges.

The bands are based on the distribution of the balance variable rather than treating each balance value as a separate category.

In [8]:
# Create broader balance categories using the observed balance values
df["balance_band"] = pd.cut(
    df["balance"],
    bins=[-np.inf, 0, 1000, 5000, 10000, np.inf],
    labels=["Negative", "0-999", "1000-4999", "5000-9999", "10000+"]
)

# Check how many customers fall into each balance band
print("Balance band distribution:")
print(df["balance_band"].value_counts().sort_index())

Balance band distribution:
balance_band
Negative      7280
0-999        23300
1000-4999    11786
5000-9999     2016
10000+         829
Name: count, dtype: int64


**3.4 Create the Repeated Contact Feature**

The `campaign` variable records the number of contacts performed during the current campaign for a customer. A binary `repeated_contact_flag` is created to identify customers who received more than one contact during the campaign.

This feature provides a simplified representation of repeated campaign contact while retaining the original `campaign` variable for the modelling stage.

In [9]:
# Identify customers who received more than one contact during the campaign
df["repeated_contact_flag"] = (df["campaign"] > 1).astype(int)

# Check the number of customers in each category
print("Repeated contact distribution:")
print(df["repeated_contact_flag"].value_counts())

# Confirm that the new feature contains only 0 and 1
print("\nUnique values:")
print(df["repeated_contact_flag"].unique())

Repeated contact distribution:
repeated_contact_flag
1    27667
0    17544
Name: count, dtype: int64

Unique values:
[0 1]


## 4. Modelling Feature Selection

Feature engineering has produced additional variables that may provide useful information for modelling. Before constructing the final modelling dataset, variables that may not be appropriate for the project's prediction objective must be considered.

**4.1 Exclude Contact Duration**

The `duration` variable records the duration of the most recent contact with the customer. This information is only known after the contact has taken place.

Including `duration` could introduce information leakage relative to an early-prediction objective, because the model would use information generated during or after the contact rather than information available before the conversion decision.

For this reason, `duration` is excluded from the primary modelling feature set. The variable remains available in the preprocessed dataset for reference but is not passed to the modelling stage.

In [11]:
# Remove duration from the primary modelling feature set
df_features = df.drop(columns=["duration"])

# Confirm that duration has been removed
print("Duration present:", "duration" in df_features.columns)

# Display the resulting dataset dimensions
print("Rows:", df_features.shape[0])
print("Columns:", df_features.shape[1])

Duration present: False
Rows: 45211
Columns: 21


**4.2 Encode Categorical Variables**

The dataset contains categorical variables that are stored as text. These variables must be converted into numerical representations before they can be used by the machine-learning models.

One-hot encoding is used to represent each category as a separate binary feature. The original categorical columns are replaced by their encoded columns.

The target variable `y_binary` is already numeric and is therefore not encoded.

In [12]:
# Identify categorical predictor variables that require one-hot encoding
categorical_columns = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "poutcome",
    "age_band",
    "balance_band"
]

# Display the categorical variables selected for encoding
print("Categorical variables to encode:")
print(categorical_columns)

Categorical variables to encode:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'age_band', 'balance_band']


One-hot encoding converts each categorical variable into a set of binary indicator columns. This allows categorical information to be represented numerically without assigning an artificial numerical order to the categories.

The `drop_first=True` option removes one category from each variable to provide a reference category and avoid redundant columns.

In [13]:
# Convert categorical variables into binary indicator columns
df_features = pd.get_dummies(
    df_features,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

# Check the dimensions after encoding
print("Rows:", df_features.shape[0])
print("Columns:", df_features.shape[1])

Rows: 45211
Columns: 53


**4.2 Validate the Encoded Dataset**

After one-hot encoding, the modelling dataset is checked to confirm that the transformation was successful.

The dataset should contain no remaining categorical predictor columns, the target should remain binary, and the number of observations should remain unchanged.

In [14]:
# Identify any remaining text-based columns
remaining_categorical = df_features.select_dtypes(include=["object", "category"]).columns.tolist()

# Check the final target values
target_values = df_features["y_binary"].unique()

# Check the number of missing values
missing_values = df_features.isna().sum().sum()

print("Remaining categorical columns:", remaining_categorical)
print("Unique target values:", target_values)
print("Total missing values:", missing_values)
print("Rows:", df_features.shape[0])
print("Columns:", df_features.shape[1])

Remaining categorical columns: ['y']
Unique target values: [0 1]
Total missing values: 36954
Rows: 45211
Columns: 53


**4.4 Remove Redundant Variables**

The original target variable `y` is removed because `y_binary` provides the numeric binary representation required for modelling.

The original `pdays` variable is also removed from the primary modelling dataset. During preprocessing, `pdays = -1` was converted to missing values because it represented the absence of a previous contact. The engineered `previously_contacted` feature now captures this information explicitly.

Removing `pdays` avoids passing a variable containing missing values into the modelling stage while retaining the information it provides through the engineered feature.

In [15]:
# Remove the original text target and the missing-value pdays variable
df_features = df_features.drop(columns=["y", "pdays"])

# Check that the variables have been removed
print("y present:", "y" in df_features.columns)
print("pdays present:", "pdays" in df_features.columns)

# Check the dimensions after removing the variables
print("Rows:", df_features.shape[0])
print("Columns:", df_features.shape[1])

y present: False
pdays present: False
Rows: 45211
Columns: 51


## 5. Final Feature-Engineered Dataset Validation

The feature-engineering stage is validated before the dataset is saved for model development.

The final dataset should contain only numerical modelling features and the binary target `y_binary`. The number of observations should remain unchanged, and there should be no missing or infinite values.

This validation confirms that the dataset is ready to be passed to the modelling stage.

In [16]:
# Identify any remaining non-numerical columns
non_numeric_columns = df_features.select_dtypes(
    exclude=["number"]
).columns.tolist()

# Count all remaining missing values
missing_values = df_features.isna().sum().sum()

# Check for infinite numerical values
infinite_values = np.isinf(
    df_features.select_dtypes(include="number")
).sum().sum()

# Confirm the target contains only 0 and 1
target_values = sorted(df_features["y_binary"].unique())

# Display the final validation results
print("Non-numerical columns:", non_numeric_columns)
print("Total missing values:", missing_values)
print("Total infinite values:", infinite_values)
print("Target values:", target_values)
print("Rows:", df_features.shape[0])
print("Columns:", df_features.shape[1])

Non-numerical columns: []
Total missing values: 0
Total infinite values: 0
Target values: [0, 1]
Rows: 45211
Columns: 51


## 6. Save the Feature-Engineered Dataset

The validated feature-engineered dataset is saved as a CSV file in the processed-data folder.

This file will serve as the input for the model-development notebooks. Saving the dataset creates a clear separation between feature engineering and model development and allows the modelling notebooks to be run independently.

In [17]:
# Define the output path for the feature-engineered dataset
FEATURES_PATH = "data/processed/bank_marketing_features.csv"

# Save the feature-engineered dataset without the Pandas index
df_features.to_csv(FEATURES_PATH, index=False)

print("Feature-engineered dataset saved successfully.")
print("Location:", FEATURES_PATH)
print("Rows:", df_features.shape[0])
print("Columns:", df_features.shape[1])

Feature-engineered dataset saved successfully.
Location: data/processed/bank_marketing_features.csv
Rows: 45211
Columns: 51


In [18]:
# Reload the saved CSV to verify that it was written correctly
check_df = pd.read_csv(FEATURES_PATH)

# Confirm the saved dataset has the expected structure
print("Saved dataset reloaded successfully.")
print("Rows:", check_df.shape[0])
print("Columns:", check_df.shape[1])
print("Missing values:", check_df.isna().sum().sum())
print("Target values:", sorted(check_df["y_binary"].unique()))

Saved dataset reloaded successfully.
Rows: 45211
Columns: 51
Missing values: 0
Target values: [0, 1]
